# Analisi dei risultati della simulazione phishing

Notebook aggiornato allo schema `decision` / `flow_outcome` / `compromise_action`.

Gli output sono simulati: le percentuali non vanno interpretate come tassi reali di successo del phishing.


## 1. Caricamento ultimo CSV o CSV selezionato

Imposta `CSV_PATH` per analizzare un file specifico. Lascia `None` per usare automaticamente l'ultimo `sim_*.csv` nella cartella `results/`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = Path('results')
MESSAGES_PATH = Path('scenarios/messages.json')
CSV_PATH = None

csv_path = Path(CSV_PATH) if CSV_PATH else max(RESULTS_DIR.glob('sim_*.csv'), key=lambda p: p.stat().st_mtime)
df = pd.read_csv(csv_path)

with MESSAGES_PATH.open('r', encoding='utf-8') as f:
    messages = {message['id']: message for message in json.load(f)}

print(f'File analizzato: {csv_path}')
print(f'Righe: {len(df)}')
if 'model' in df.columns:
    print('Modello:', ', '.join(sorted(df['model'].dropna().astype(str).unique())))
if 'temperature' in df.columns:
    print('Temperature:', ', '.join(sorted(df['temperature'].dropna().astype(str).unique())))


## Normalizzazione minima

Le colonne del nuovo schema sono usate come fonte principale. Gli alias storici vengono letti solo come fallback per analizzare vecchi CSV.


In [ ]:
PHISHING_TYPE = 'phishing'
LEGITIMATE_TYPE = 'legittimo'

DECISION_PROCEED = 'PROCEDE_CON_LA_RICHIESTA'
DECISION_VERIFY = 'VERIFICA_TRAMITE_CANALE_UFFICIALE'
DECISION_REPORT = 'SEGNALA_COME_PHISHING'
DECISION_IGNORE = 'IGNORA'
DECISION_DELAY = 'RIMANDA_O_NON_DECIDE'

FLOW_NO_ENTRY = 'NON_ENTRA_NEL_FLOW'
FLOW_STOPPED = 'SI_FERMA_PRIMA_DELLA_COMPROMISSIONE'
FLOW_COMPROMISED = 'COMPROMISSIONE_COMPLETATA'
FLOW_LEGITIMATE = 'AZIONE_LEGITTIMA_COMPLETATA'
NO_COMPROMISE_ACTION = 'NESSUNA'


def is_true(series):
    return series.astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'si'})


def infer_archetype(agent_id):
    agent_id = str(agent_id)
    if '_' not in agent_id:
        return agent_id
    prefix, suffix = agent_id.rsplit('_', 1)
    return prefix if suffix.isdigit() else agent_id

work = df.copy()
work['message_type'] = work.get('message_type', '').astype(str).str.strip().str.lower().replace({
    'legitimo': LEGITIMATE_TYPE,
    'legitimate': LEGITIMATE_TYPE,
    'legittima': LEGITIMATE_TYPE,
})

if 'decision' not in work.columns and 'initial_reaction' in work.columns:
    work['decision'] = work['initial_reaction']
if 'flow_outcome' not in work.columns:
    work['flow_outcome'] = ''
if 'compromise_action' not in work.columns:
    if 'specific_action' in work.columns:
        work['compromise_action'] = work['specific_action']
    elif 'final_action' in work.columns:
        work['compromise_action'] = work['final_action']
    else:
        work['compromise_action'] = NO_COMPROMISE_ACTION

if 'entered_flow' not in work.columns:
    work['entered_flow'] = work['decision'].eq(DECISION_PROCEED)
else:
    work['entered_flow'] = is_true(work['entered_flow'])

if 'stopped_before_compromise' not in work.columns:
    work['stopped_before_compromise'] = work['flow_outcome'].eq(FLOW_STOPPED)
else:
    work['stopped_before_compromise'] = is_true(work['stopped_before_compromise'])

for col, decision_label in [
    ('verified', DECISION_VERIFY),
    ('reported', DECISION_REPORT),
    ('ignored', DECISION_IGNORE),
    ('delayed', DECISION_DELAY),
]:
    if col not in work.columns:
        work[col] = work['decision'].eq(decision_label)
    else:
        work[col] = is_true(work[col])

if 'legitimate_completion' not in work.columns:
    work['legitimate_completion'] = work['flow_outcome'].eq(FLOW_LEGITIMATE)
else:
    work['legitimate_completion'] = is_true(work['legitimate_completion'])

if 'compromised' not in work.columns:
    work['compromised'] = False
else:
    work['compromised'] = is_true(work['compromised'])

if 'parse_error' not in work.columns:
    work['parse_error'] = False
else:
    work['parse_error'] = is_true(work['parse_error'])

if 'validation_error' not in work.columns:
    work['validation_error'] = ''
else:
    work['validation_error'] = work['validation_error'].fillna('').astype(str)

if 'archetype_id' not in work.columns:
    work['archetype_id'] = work['agent_id'].apply(infer_archetype)

work['flow_outcome'] = work['flow_outcome'].fillna('').replace('', FLOW_NO_ENTRY)
work['compromise_action'] = work['compromise_action'].fillna('').replace('', NO_COMPROMISE_ACTION)


## 2. Controllo qualita

Questa cella verifica righe, parse error, validation error, vecchie label e compromissioni sui messaggi legittimi.


In [ ]:
forbidden_labels = ['APRE_' + 'MESSAGGIO_O_LINK', 'CLICCA_' + 'LINK_SENZA_INSERIRE_DATI']
old_label_hits = {
    label: work.astype(str).apply(lambda col: col.str.contains(label, regex=False, na=False)).any().any()
    for label in forbidden_labels
}
legitimate_df = work[work['message_type'].eq(LEGITIMATE_TYPE)]
phishing_df = work[work['message_type'].eq(PHISHING_TYPE)]

quality = pd.DataFrame([
    {'controllo': 'righe', 'valore': len(work)},
    {'controllo': 'parse_error', 'valore': int(work['parse_error'].sum())},
    {'controllo': 'validation_error', 'valore': int(work['validation_error'].str.strip().ne('').sum())},
    {'controllo': 'legittimi_compromised', 'valore': int(legitimate_df['compromised'].sum())},
    *[{'controllo': f'label_obsoleta_{label}', 'valore': bool(hit)} for label, hit in old_label_hits.items()],
])
quality


## 3. Distribuzione decision


In [ ]:
decision_distribution = work['decision'].value_counts(dropna=False).rename_axis('decision').reset_index(name='count')
decision_distribution['rate'] = decision_distribution['count'] / len(work)
decision_distribution


## 4. Distribuzione flow_outcome


In [ ]:
flow_distribution = work['flow_outcome'].value_counts(dropna=False).rename_axis('flow_outcome').reset_index(name='count')
flow_distribution['rate'] = flow_distribution['count'] / len(work)
flow_distribution


## 5. Distribuzione compromise_action


In [ ]:
action_distribution = work['compromise_action'].value_counts(dropna=False).rename_axis('compromise_action').reset_index(name='count')
action_distribution['rate'] = action_distribution['count'] / len(work)
action_distribution


## 6. Metriche phishing

`entered_flow` non equivale a compromissione. La compromissione richiede phishing, flow completato e azione compatibile con lo scenario.


In [ ]:
phishing_metrics = pd.DataFrame([
    {'metrica': 'entered_flow', 'count': int(phishing_df['entered_flow'].sum()), 'total': len(phishing_df)},
    {'metrica': 'stopped_before_compromise', 'count': int(phishing_df['stopped_before_compromise'].sum()), 'total': len(phishing_df)},
    {'metrica': 'compromised', 'count': int(phishing_df['compromised'].sum()), 'total': len(phishing_df)},
])
phishing_metrics['rate'] = phishing_metrics['count'] / phishing_metrics['total'].replace(0, pd.NA)
phishing_metrics


## 7. Metriche legittimi

I messaggi legittimi dovrebbero avere `compromised = False`.


In [ ]:
legitimate_metrics = pd.DataFrame([
    {'metrica': 'legitimate_completion', 'count': int(legitimate_df['legitimate_completion'].sum()), 'total': len(legitimate_df)},
    {'metrica': 'verified', 'count': int(legitimate_df['verified'].sum()), 'total': len(legitimate_df)},
    {'metrica': 'ignored', 'count': int(legitimate_df['ignored'].sum()), 'total': len(legitimate_df)},
    {'metrica': 'delayed', 'count': int(legitimate_df['delayed'].sum()), 'total': len(legitimate_df)},
    {'metrica': 'compromised', 'count': int(legitimate_df['compromised'].sum()), 'total': len(legitimate_df)},
])
legitimate_metrics['rate'] = legitimate_metrics['count'] / legitimate_metrics['total'].replace(0, pd.NA)
legitimate_metrics


## 8. Analisi per scenario


In [ ]:
def rate(series):
    return series.mean() if len(series) else 0

scenario_summary = (
    work.groupby(['message_id', 'message_type'], dropna=False)
    .agg(
        n=('message_id', 'size'),
        entered_flow=('entered_flow', rate),
        stopped_before_compromise=('stopped_before_compromise', rate),
        compromised=('compromised', rate),
        verified=('verified', rate),
        reported=('reported', rate),
        ignored=('ignored', rate),
        delayed=('delayed', rate),
        legitimate_completion=('legitimate_completion', rate),
        parse_error=('parse_error', rate),
    )
    .reset_index()
)
scenario_summary


## 9. Analisi per archetipo


In [ ]:
archetype_summary = (
    work.groupby('archetype_id', dropna=False)
    .agg(
        n=('archetype_id', 'size'),
        entered_flow=('entered_flow', rate),
        stopped_before_compromise=('stopped_before_compromise', rate),
        compromised=('compromised', rate),
        verified=('verified', rate),
        reported=('reported', rate),
        ignored=('ignored', rate),
        delayed=('delayed', rate),
        legitimate_completion=('legitimate_completion', rate),
        parse_error=('parse_error', rate),
    )
    .reset_index()
)
archetype_summary


## 10. Tabella scenario x decision


In [ ]:
scenario_decision = pd.crosstab(work['message_id'], work['decision'])
scenario_decision


In [ ]:
plt.figure(figsize=(12, max(4, len(scenario_decision) * 0.45)))
sns.heatmap(scenario_decision, annot=True, fmt='d', cmap='Blues')
plt.title('Scenario x decision')
plt.xlabel('decision')
plt.ylabel('message_id')
plt.tight_layout()
plt.show()


## 11. Tabella scenario x flow_outcome


In [ ]:
scenario_flow = pd.crosstab(work['message_id'], work['flow_outcome'])
scenario_flow


In [ ]:
plt.figure(figsize=(12, max(4, len(scenario_flow) * 0.45)))
sns.heatmap(scenario_flow, annot=True, fmt='d', cmap='Greens')
plt.title('Scenario x flow_outcome')
plt.xlabel('flow_outcome')
plt.ylabel('message_id')
plt.tight_layout()
plt.show()


## 12. Limiti metodologici

Questa analisi descrive una simulazione, non un esperimento su utenti reali. Gli agenti sono archetipi sintetici e il modello puo avere bias prudenziali, bias di sicurezza o tendenze a risposte troppo regolari.

Le percentuali dipendono da prompt, modello, temperatura, scenari e regole di validazione. Eventuali pattern come poche segnalazioni, molte verifiche o molti stop prima della compromissione vanno discussi come limiti metodologici e non come distribuzioni da forzare.

Il confronto piu utile e tra versioni controllate della simulazione: stesso schema, stessi scenari, stesso modello o cambiamenti documentati.
